## Шаг 2. Выбор цели, даты и группы

В рамках разведочного анализа набора данных о перелетах во время пандемии COVID-19 были выбраны следующие переменные:

*   **Целевая переменная:** `altitude_1` (высота полета в футах). 
    *Обоснование:* Это непрерывная числовая величина, которая характеризует параметры полета. Анализ её распределения позволит выявить типичные высоты, а также аномалии (например, полеты на нестандартно низких высотах во время посадки или на очень больших высотах).
*   **Столбец даты:** `day`.
    *Обоснование:* Данные охватывают период пандемии, когда авиасообщение сильно менялось. Период агрегации выбран **день**, так как это позволит детально отследить динамику изменения количества рейсов в моменты введения локдаунов и ограничений.
*   **Столбец группы:** `origin` (аэропорт вылета).
    *Обоснование:* Это категориальный признак с повторяющимися значениями (коды аэропортов IATA/ICAO). В ходе очистки данных (Лабораторная работа 1) строки с отсутствующим аэропортом вылета были удалены, поэтому в текущем наборе этот столбец не содержит пропусков. Группировка по аэропорту позволит сравнить, из каких хабов совершается больше всего рейсов и каковы их типичные параметры.

In [2]:
import pandas as pd

# Загружаем данные
df = pd.read_parquet("../data/processed/cleanfull.parquet")

# Проверяем выбранные столбцы
print("Проверка данных:")
print(f"Пропуски в altitude_1: {df['altitude_1'].isna().sum()}")
print(f"Пропуски в day: {df['day'].isna().sum()}")
print(f"Пропуски в origin: {df['origin'].isna().sum()}")  # Должно быть 0!

# Смотрим, как выглядят данные
print(df[["altitude_1", "day", "origin"]].head())

Проверка данных:
Пропуски в altitude_1: 410954
Пропуски в day: 0
Пропуски в origin: 0
   altitude_1                       day origin
0  304.799988 2019-01-01 00:00:00+00:00   YMML
1  304.799988 2019-01-01 00:00:00+00:00   YMML
2    0.000000 2019-01-01 00:00:00+00:00   YSSY
3  609.599976 2019-01-01 00:00:00+00:00   LEMD
4    0.000000 2019-01-01 00:00:00+00:00   YSSY


In [3]:
# Шаг 3. Распределение целевой переменной (altitude_1)

# Берем столбец с целевой переменной
target = df['altitude_1']

# Считаем статистику
count_total = len(target)
count_nan = target.isna().sum()
min_val = target.min()
q1 = target.quantile(0.25)
median_val = target.median()
mean_val = target.mean()
q3 = target.quantile(0.75)
max_val = target.max()

# Определяем скошенность (сравниваем среднее и медиану)
skew_text = "вправо (среднее больше медианы)" if mean_val > median_val else "влево (среднее меньше медианы)"

# Выводим результат в консоль (чтобы ты могла заполнить таблицу)
print(f"Число наблюдений: {count_total}")
print(f"Число пропусков: {count_nan}")
print(f"Минимум: {min_val:.2f}")
print(f"Первый квартиль: {q1:.2f}")
print(f"Медиана: {median_val:.2f}")
print(f"Среднее: {mean_val:.2f}")
print(f"Третий квартиль: {q3:.2f}")
print(f"Максимум: {max_val:.2f}")
print(f"Скошенность: {skew_text}")

Число наблюдений: 7248316
Число пропусков: 410954
Минимум: 0.00
Первый квартиль: 0.00
Медиана: 304.80
Среднее: 424.27
Третий квартиль: 609.60
Максимум: 5791.20
Скошенность: вправо (среднее больше медианы)


### Таблица 1 - Сводка целевой переменной

| Показатель | Значение |
| :--- | :--- |
| Число наблюдений | 7248316 |
| Число пропусков | 410 954 |
| Минимум | 0 |
| Первый квартиль | 0 |
| Медиана | 304.80 |
| Среднее | 424.27 |
| Третий квартиль | 609.60 |
| Максимум | 5791.20 |
| Скошенность | вправо (среднее больше медианы) |

Пояснение: «Распределение высоты полета скошено вправо. Это означает, что большинство рейсов летает на средних высотах, но есть небольшая часть полетов на очень больших высотах, которые тянут среднее значение вверх, делая его больше медианы».

In [4]:
# Шаг 4. Динамика по датам

import pandas as pd

# 1. Оставляем только строки, где есть дата, и группируем по дням
df_daily = df.dropna(subset=['day']).groupby('day').agg(
    count=('altitude_1', 'size'),          # Число записей
    median_target=('altitude_1', 'median'), # Медиана цели
    mean_target=('altitude_1', 'mean')      # Среднее цели
).reset_index()

# 2. Создаем полный календарь (от мин до макс даты), чтобы вставить пустые дни
min_date = df_daily['day'].min()
max_date = df_daily['day'].max()
full_dates = pd.date_range(start=min_date, end=max_date, freq='D')

# Переиндексируем по полному календарю (пустые дни станут NaN)
df_daily = df_daily.set_index('day').reindex(full_dates).reset_index()
df_daily.rename(columns={'index': 'day'}, inplace=True)

# 3. Заполняем пустые периоды нулями для числа записей
df_daily['count'] = df_daily['count'].fillna(0).astype(int)

# 4. Выводим данные: Начало, Середина, Конец
print("--- НАЧАЛО РЯДА ---")
print(df_daily.head(5).to_string(index=False))

print("\n--- СЕРЕДИНА РЯДА ---")
middle_idx = len(df_daily) // 2
print(df_daily.iloc[middle_idx-2:middle_idx+3].to_string(index=False))

print("\n--- КОНЕЦ РЯДА ---")
print(df_daily.tail(5).to_string(index=False))

--- НАЧАЛО РЯДА ---
                      day  count  median_target  mean_target
2019-01-01 00:00:00+00:00  28555     304.799988   406.955780
2019-01-02 00:00:00+00:00  36210     304.799988   426.363037
2019-01-03 00:00:00+00:00  38172     304.799988   442.004639
2019-01-04 00:00:00+00:00  40450     304.799988   457.143677
2019-01-05 00:00:00+00:00  35983     304.799988   468.234650

--- СЕРЕДИНА РЯДА ---
                      day  count  median_target  mean_target
2019-03-30 00:00:00+00:00  33411     304.799988   412.229340
2019-03-31 00:00:00+00:00  35064     304.799988   412.777191
2019-04-01 00:00:00+00:00  38950     304.799988   413.472778
2019-04-02 00:00:00+00:00  38280     304.799988   426.315918
2019-04-03 00:00:00+00:00  44142     304.799988   423.220886

--- КОНЕЦ РЯДА ---
                      day  count  median_target  mean_target
2019-06-26 00:00:00+00:00  50862     304.799988   432.767639
2019-06-27 00:00:00+00:00  47633     304.799988   438.504364
2019-06-28 00:00:00+00

C:\Users\Jesayz\AppData\Local\Temp\ipykernel_20220\3727342268.py:19: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_daily.rename(columns={'index': 'day'}, inplace=True)


### Таблица 2 - Динамика по периодам

| Период (Дата) | Число записей | Медиана цели | Среднее цели |
| :--- | :--- | :--- | :--- |
| 2019-01-01 | 28555 | 304.799988 | 406.955780 |
| 2019-01-05 | 35983 | 304.799988 | 468.234650 |
| 2019-04-01 | 38950 | 304.799988 | 413.472778 |
| 2019-06-26 | 50862 | 304.799988 | 432.767639 |
| 2019-06-30 | 44507 | 304.799988 | 412.634796 |

In [5]:
# Шаг 5. Агрегаты по группам и сводная таблица

# 1. Считаем агрегаты по каждой группе (аэропорту вылета)
group_stats = df.groupby('origin').agg(
    count=('altitude_1', 'size'),              # Число записей в группе
    median=('altitude_1', 'median'),           # Медиана высоты
    mean=('altitude_1', 'mean'),               # Среднее высоты
    missing_share=('altitude_1', lambda x: x.isna().mean() * 100) # Доля пропусков в %
).reset_index()

# Сортируем по числу записей (чтобы самые частые аэропорты были сверху)
group_stats = group_stats.sort_values(by='count', ascending=False)

# Выведем ТОП-5 и ТОП-5 самых редких для таблицы 3 в отчет
print("--- ТОП-5 аэропортов по числу рейсов ---")
print(group_stats.head(5).to_string(index=False))

print("\n--- 5 самых редких аэропортов ---")
print(group_stats.tail(5).to_string(index=False))

--- ТОП-5 аэропортов по числу рейсов ---
origin  count     median       mean  missing_share
  KORD 119432 304.799988 406.035889       0.000000
  KLAX 115942   0.000000 147.005600       0.631350
  EGLL  92045   0.000000   9.215900      23.501549
  KLAS  91947 609.599976 566.439270       0.000000
  KDFW  90923 304.799988 203.819061       0.000000

--- 5 самых редких аэропортов ---
origin  count      median        mean  missing_share
  YLSM     10 1524.000000 1463.040039            0.0
  09IN     10  609.599976  640.079956            0.0
  OK49     10 1828.800049 1920.239990            0.0
  CNX8     10 1371.599976 1371.599976            0.0
  GA73     10 1524.000000 1402.079956            0.0


In [6]:
# Создаем столбец с месяцем (например, '2020-03') для сводной таблицы
df['month'] = df['day'].dt.to_period('M').astype(str)

# Берем топ-10 самых частых аэропортов (чтобы таблица не была гигантской)
top_airports = group_stats.head(10)['origin'].tolist()
df_top = df[df['origin'].isin(top_airports)]

# Строим сводную таблицу (строки - аэропорты, столбцы - месяцы, значения - медиана высоты)
pivot_table = pd.pivot_table(
    df_top, 
    values='altitude_1', 
    index='origin', 
    columns='month', 
    aggfunc='median'
)

print("--- Сводная таблица (Медиана высоты по аэропортам и месяцам) ---")
print(pivot_table.to_string())

C:\Users\Jesayz\AppData\Local\Temp\ipykernel_20220\1838493907.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['month'] = df['day'].dt.to_period('M').astype(str)


--- Сводная таблица (Медиана высоты по аэропортам и месяцам) ---
month       2019-01      2019-02      2019-03      2019-04      2019-05      2019-06
origin                                                                              
EDDF       0.000000     0.000000     0.000000     0.000000     0.000000     0.000000
EGLL       0.000000     0.000000     0.000000     0.000000     0.000000     0.000000
EHAM       0.000000     0.000000     0.000000     0.000000     0.000000     0.000000
KDEN    1524.000000  1524.000000  1524.000000  1828.800049  1828.800049  1828.800049
KDFW       0.000000     0.000000   304.799988   304.799988   304.799988   304.799988
KLAS     609.599976   609.599976   609.599976   609.599976   609.599976   609.599976
KLAX       0.000000     0.000000     0.000000     0.000000   304.799988   304.799988
KORD     304.799988   304.799988   304.799988   304.799988   304.799988   609.599976
KSFO       0.000000     0.000000     0.000000     0.000000     0.000000     0.000000


### Таблица 3 - Агрегаты по группам (Аэропорты вылета)

| Группа (Аэропорт) | Число | Медиана | Среднее | Доля пропусков (%) |
| :--- | :--- | :--- | :--- | :--- |
| KORD | 119432 | 304.799988 | 406.035889 | 0 |
| KLAX | 115942 | 0 | 147.005600 | 0.631350 |
| EGLL | 92045 | 0 | 9.215900 | 23.501549 |
| KLAS | 91947 | 609.599976 | 566.439270 | 0 |
| KDFW | 90923 | 304.799988 | 203.819061 | 0 |

**Вывод по агрегатам и сводной таблице:**

Наибольший объем данных приходится на крупные международные хабы: KORD (Чикаго), KLAX (Лос-Анджелес), EGLL (Лондон), KLAS (Лас-Вегас) и KDFW (Даллас). Число записей в этих группах колеблется от 90 до 119 тысяч.

При анализе медианной высоты (`median`) обнаружена важная особенность данных: у аэропортов KLAX и EGLL медиана равна 0.0. Это означает, что более половины записей в этих группах содержат нулевую высоту (вероятно, данные о полетах на малых высотах или ошибки фиксации, характерные для этих зон). При этом у аэропортов KORD, KLAS и KDFW медиана находится на реалистичном уровне (около 300–600 футов). 

Аэропорт EGLL также выделяется высокой долей пропусков в данных о высоте — 23.5%. У остальных лидеров пропусков почти нет.

**По сводной таблице:** 
Медианные высоты для аэропорта KDEN (Денвер) стабильны по месяцам (около 1524–1828 футов). Аэропорт KLAS показывает стабильную медиану в 609.59 футов на протяжении всех месяцев. В то же время для многих аэропортов (например, EDDF, EGLL, LFPG) в период с января по май 2019 года медиана равна 0.0. Это может свидетельствовать о том, что в этот период в данных преобладали записи с нулевыми значениями высоты, либо данные за эти месяцы неполные.

In [7]:
# Шаг 6. Аномалии

import sys
sys.path.append('..') # Чтобы блокнот видел папку src
from src.eda import get_anomaly_masks, print_anomaly_stats

# Применяем функцию к нашему датафрейму
mask_iqr, mask_business, mask_intersection = get_anomaly_masks(df, 'altitude_1')

# Выводим статистику
print_anomaly_stats(df, mask_iqr, mask_business, mask_intersection)

Всего строк: 7248316
Межквартильный размах (IQR): 279628 (3.86%)
Второе правило (высота < 0): 0 (0.00%)
Пересечение: 0 (0.00%)


### Таблица 4 - Сопоставление правил аномалий

| Правило | Число отмеченных | Доля (%) | Совпало со вторым |
| :--- | :--- | :--- | :--- |
| Межквартильный размах (IQR) | 279 628 | 3.86% | 0 |
| Второе правило (высота < 0) | 0 | 0.00% | 0 |
| Пересечение | 0 | 0.00% | — |

**Вывод по аномалиям:**

Для поиска аномалий были выбраны два правила:
1. **Межквартильный размах (IQR):** отмечает значения, которые лежат за пределами 1.5 * IQR от первого и третьего квартилей.
2. **Предметное ограничение (высота < 0):** физически невозможно, чтобы самолет летел на отрицательной высоте.

Правило IQR отметило 279 628 строк (3.86% от всего набора). Это ожидаемый результат, так как распределение высот имеет длинный «хвост» вправо (большие высоты), которые алгоритм IQR считает выбросами. 

Второе правило (высота < 0) отметило 0 строк. Это связано с тем, что на этапе очистки (Лабораторная работа 1) все некорректные отрицательные высоты уже были заменены на пропуски (NaN). Пересечение списков также составило 0 строк, что подтверждает отсутствие грубых ошибок в данных. Файл cleanfull.parquet при этом не изменялся.